# VERSION 2

# E1: PRETRAITEMENT DE L'AUDIO

In [ ]:
import librosa
import numpy as np
import io

def preparer_audio(source_audio, target_sr=16000, is_bytes=False, original_sr_bytes=None):
    """
    Charge, rééchantillonne, passe en mono et normalise un signal audio,
    qu'il provienne d'un fichier local ou d'un flux d'octets (Streamlit).
    """
    # 1. Chargement de l'audio selon la source
    if is_bytes:
        if original_sr_bytes is None:
            raise ValueError("Le taux d'échantillonnage d'origine (original_sr_bytes) doit être fourni pour les octets bruts.")

        # Si la source est un tableau numpy d'octets convertis
        y = source_audio.astype(np.float32)
        sr = original_sr_bytes

        # Passage en mono manuel si stéréo (2 canaux)
        if len(y.shape) > 1:
            y = np.mean(y, axis=1)
    else:
        # Chargement classique depuis un chemin de fichier (ESC-50)
        y, sr = librosa.load(source_audio, sr=None, mono=True)

    # 2. Rééchantillonnage strict à 16000 Hz
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    # 3. Normalisation Robuste (Remplacement du Peak Normalization)
    # Remplacement par une normalisation de type RMS ou standardisation
    max_val = np.max(np.abs(y))
    if max_val > 0:
        y_norm = y / max_val
    else:
        y_norm = y

    # 4. Forcer la longueur stricte de 5 secondes (Spécifique ESC-50)
    # 5 secondes à 16kHz = 80 000 échantillons
    duree_cible = target_sr * 5
    if len(y_norm) < duree_cible:
        # Remplissage par du vide si trop court
        y_norm = np.pad(y_norm, (0, duree_cible - len(y_norm)), 'constant')
    elif len(y_norm) > duree_cible:
        # Tronquage si trop long
        y_norm = y_norm[:duree_cible]

    return y_norm, sr


# EXTRACTION DES EMBEDINGS

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

# 1. Chargement du modèle YAMNet depuis TensorFlow Hub
print("Chargement du modèle YAMNet depuis TF Hub...")
yamnet_model_handle = 'https://tfhub.dev/google/yamnet/1'
yamnet_model = hub.load(yamnet_model_handle)
print("Modèle YAMNet chargé avec succès !")

def extraire_embeddings(source_audio, is_bytes=False, original_sr_bytes=None):
    """
    Passe l'audio (fichier ou tableau numpy/bytes) dans le prétraitement,
    puis extrait l'embedding moyen pour le classifieur personnalisé.

    Retourne également les scores natifs de YAMNet pour servir de filtre de confiance.
    """
    # 1. Appel de notre fonction de pré-traitement mise à jour à l'étape précédente
    y_norm, sr = preparer_audio(
        source_audio,
        target_sr=16000,
        is_bytes=is_bytes,
        original_sr_bytes=original_sr_bytes
    )

    # 2. YAMNet attend un tenseur 1D de type float32
    waveform = tf.convert_to_tensor(y_norm, dtype=tf.float32)

    # 3. Inférence avec YAMNet
    # scores: (N_tranches, 521 classe nativas)
    # embeddings: (N_tranches, 1024)
    scores, embeddings, spectrogram = yamnet_model(waveform)

    # 4. Gestion d'un flux totalement muet (Sécurité pour Streamlit)
    if embeddings.shape[0] == 0:
        # Si YAMNet ne génère aucune frame (signal vide), on renvoie un vecteur de zéros
        return np.zeros(1024, dtype=np.float32), np.zeros(521, dtype=np.float32)

    # 5. Calcul de la représentation globale (Moyenne temporelle)
    embedding_moyen = tf.reduce_mean(embeddings, axis=0).numpy()

    # 6. Extraction du score moyen natif de YAMNet (Optionnel mais recommandé)
    score_moyen_native = tf.reduce_mean(scores, axis=0).numpy()

    return embedding_moyen, score_moyen_native


Chargement du modèle YAMNet depuis TF Hub...
Modèle YAMNet chargé avec succès !


Extraction des classes visées et gestions de bruits d'environnement

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import json

# 1. Définition des chemins avec le sous-dossier ESC-50-master
dossier_esc50 = '/content/drive/MyDrive/ESC-50'
dossier_audio = os.path.join(dossier_esc50, 'ESC-50-master', 'audio')
csv_metadonnees = os.path.join(dossier_esc50, 'ESC-50-master', 'meta', 'esc50.csv')

# 2. Chargement du fichier CSV officiel
df_meta = pd.read_csv(csv_metadonnees)

# 3. FILTRAGE STRICT POUR NE GARDER QUE LES BRUITS DOMESTIQUES ET QUOTIDIENS
classes_domestiques_cibles = [
    'crying_baby',           # Bébé qui pleure
    'siren',              # Sirène
    'clapping',           # Applaudissements
    'door_wood_knock',    # Toc-toc à la porte
    'glass_breaking',     # Verre brisé
    'car_horn',           # Klaxon de voiture
    'pouring_water',      # Gouttes d'eau / fuite
    'washing_machine',    # Machine à laver
    'crackling_fire',      # Crépitement de feu
    'clock_alarm'         # Alarme de réveil
]

# Filtrage du DataFrame
df_filtre = df_meta[df_meta['category'].isin(classes_domestiques_cibles)].copy()

# Création d'un dictionnaire de mapping fixe (ex: 'baby_cry' -> 0, 'clock_alarm' -> 1, etc.)
classes_uniques = sorted(df_filtre['category'].unique())  # Trié pour garder un ordre strict au redémarrage
mapping_classes = {classe: idx for idx, classe in enumerate(classes_uniques)}
df_filtre['target_label'] = df_filtre['category'].map(mapping_classes)

print(f"Nombre total de fichiers de bruits domestiques sélectionnés : {len(df_filtre)}")
print(f"Classes incluses ({len(classes_uniques)} classes) :")
for nom, idx in mapping_classes.items():
    print(f" - ID {idx} : {nom}")

# Sauvegarder impérativement le mapping dans Google Drive !
chemin_mapping_drive = "/content/drive/MyDrive/mapping_classes.json"
with open(chemin_mapping_drive, "w") as f:
    json.dump(mapping_classes, f)

print("Mapping sauvegardé avec succès dans le Drive :", chemin_mapping_drive)

# 4. BOUCLE D'EXTRACTION DES EMBEDDINGS YAMNET AVEC SÉCURITÉ
X_embeddings = []
y_labels = []

print("\nDébut de l'extraction des embeddings YAMNet...")
for index, row in df_filtre.iterrows():
    nom_fichier = row['filename']
    label_id = row['target_label']

    chemin_complet = os.path.join(dossier_audio, nom_fichier)

    if os.path.exists(chemin_complet):
        try:
            # Utilisation sécurisée de la fonction d'extraction
            emb, _ = extraire_embeddings(chemin_complet, is_bytes=False)
            X_embeddings.append(emb)
            y_labels.append(label_id)
        except Exception as e:
            # Si un fichier WAV spécifique est corrompu ou illisible, on passe au suivant
            print(f"⚠️ Fichier ignoré car corrompu ou illisible : {nom_fichier} | Erreur : {e}")
    else:
        print(f"Alerte : Fichier introuvable -> {chemin_complet}")

# Conversion en tableaux NumPy
X_data = np.array(X_embeddings)
y_data = np.array(y_labels)

# --- MODIFICATION CRUCIALE POUR LA COUCHE SIGMOID EN SORTIE ---
# Encodage en One-Hot pour correspondre à la fonction de perte 'binary_crossentropy'
y_data_onehot = to_categorical(y_data, num_classes=len(classes_uniques))

print(f"\nForme finale de la matrice X_data (Embeddings) : {X_data.shape}")
print(f"Forme finale de la matrice y_data_onehot (One-Hot) : {y_data_onehot.shape}")

Nombre total de fichiers de bruits domestiques sélectionnés : 400
Classes incluses (10 classes) :
 - ID 0 : car_horn
 - ID 1 : clapping
 - ID 2 : clock_alarm
 - ID 3 : crackling_fire
 - ID 4 : crying_baby
 - ID 5 : door_wood_knock
 - ID 6 : glass_breaking
 - ID 7 : pouring_water
 - ID 8 : siren
 - ID 9 : washing_machine
Mapping sauvegardé avec succès dans le Drive : /content/drive/MyDrive/mapping_classes.json

Début de l'extraction des embeddings YAMNet...


/tmp/ipykernel_2631/2260703495.py:24: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(source_audio, sr=None, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


⚠️ Fichier ignoré car corrompu ou illisible : 2-89516-A-37.wav | Erreur : 

Forme finale de la matrice X_data (Embeddings) : (399, 1024)
Forme finale de la matrice y_data_onehot (One-Hot) : (399, 10)
